# Cuaderno 01: Auditoría de Ingesta, Supervivencia de Enlaces y Respaldo Tabular

## 1. Planteamiento del Problema: Degradación de Enlaces y Sesgo de Supervivencia
En conjuntos de datos públicos de redes sociales (TikTok, Instagram Reels y YouTube Shorts), una fracción importante de las publicaciones originales deja de estar disponible con el paso del tiempo debido a eliminación por parte del autor, cambio a cuentas privadas o restricciones de derechos de autor.

Desde el punto de vista estadístico, esta pérdida de datos **no ocurre de manera completamente aleatoria** (*Missing Not At Random - MNAR*). Los creadores eliminan con mayor probabilidad aquellos videos que obtuvieron baja interacción inicial, conservando únicamente los exitosos. Si el sistema entrenara modelos solo con el subconjunto sobreviviente cuando la pérdida supera el **30 %** (`UMBRAL_TASA_CAIDOS = 0.30`), incurriría en **sesgo de supervivencia** (*Survivorship Bias*), distorsionando la distribución real del `Engagement Rate`.


In [1]:
import pandas as pd
from config import settings
from src.ingesta.cargador_kaggle import cargar_dataset_tiktok, cargar_dataset_respaldo
from src.ingesta.gestor_lotes import cargar_historial_fallos, UMBRAL_CAIDOS_MAX
from src.ingesta.validador_medios import es_archivo_valido

df_tiktok = cargar_dataset_tiktok()
df_fallos = cargar_historial_fallos()
archivos_medios = [
    p for p in sorted(settings.DATOS_MEDIOS_DIR.glob('*'))
    if p.suffix.lower() in {'.mp3', '.wav', '.ogg', '.flac'}
]
medios_validos = [p for p in archivos_medios if es_archivo_valido(p)[0]]

print(f'Total de publicaciones indexadas en TikTok crudo: {len(df_tiktok)}')
print(f'Enlaces registrados en bitacora de fallos (fallos_descarga.csv): {len(df_fallos)}')
print(f'Archivos de audio validos disponibles en datos/medios/: {len(medios_validos)}')
print(f'Umbral critico de activacion de contingencia: {UMBRAL_CAIDOS_MAX * 100:.1f}%')
print('\nMuestra de bitacora de enlaces caidos:')
print(df_fallos[['id_publicacion', 'fecha_intento', 'motivo_error']].head(5).to_string(index=False))


Total de publicaciones indexadas en TikTok crudo: 299
Enlaces registrados en bitacora de fallos (fallos_descarga.csv): 7
Archivos de audio validos disponibles en datos/medios/: 10
Umbral critico de activacion de contingencia: 30.0%

Muestra de bitacora de enlaces caidos:
     id_publicacion       fecha_intento                                                                                                                                                                                               motivo_error
6999919482068077826 2026-11-12 23:34:47 Cabecera de audio corrupta o formato no decodificable: Error opening '/home/zfernandez/Documents/5toSemestre/InteligenciaArtificial/Proyecto/datos/medios/6999919482068077826.m4a': Format not recognised.
6989072033573391621 2026-11-12 23:34:48               Error de extraccion yt-dlp: ERROR: [TikTok] 6989072033573391621: You do not have permission to view this post. Log into an account that has access. Use --cookies-from-browser or --cookies 

### Análisis de Resultados de Supervivencia de Enlaces
La bitácora persistente `datos/fallos_descarga.csv` demuestra que las URLs históricas de TikTok presentan bloqueos por permisos de cuenta privada (`You do not have permission to view this post`) o restricciones regionales/eliminación. Al superar la cota del 30 % definida en `config/settings.py`, el módulo `src/ingesta/gestor_lotes.py` evita reintentos redundantes sobre enlaces caídos y activa de forma determinista la integración con el dataset oficial de contingencia (`yamqwe/tiktok-trending-tracks`).


In [2]:
from src.ingesta.scraper_tendencias import consultar_tendencias_locales

df_matriz = pd.read_parquet(settings.RUTA_MATRIZ_CRUDA)
nulos_totales = int(df_matriz.isna().sum().sum())

print(f'Dimensiones de matriz_acustica_cruda.parquet: {df_matriz.shape[0]} filas x {df_matriz.shape[1]} columnas')
print(f'Conteo total de valores nulos: {nulos_totales}')
print('\nResumen estadistico de las 4 metricas sociales homologadas:')
print(df_matriz[settings.COLUMNAS_SOCIALES].describe().round(2).to_string())

df_tendencias = consultar_tendencias_locales(top_n=3)
print('\nMuestra del catalogo multi-plataforma (TikTok e Instagram Reels):')
print(df_tendencias.to_string(index=False))


Dimensiones de matriz_acustica_cruda.parquet: 650 filas x 33 columnas
Conteo total de valores nulos: 0

Resumen estadistico de las 4 metricas sociales homologadas:
              views        likes   comments     shares
count  6.500000e+02       650.00     650.00     650.00
mean   8.172724e+05     88407.72    4961.81    6660.54
std    5.826442e+06    606737.99   16607.37   44684.98
min    5.160000e+02        54.00       7.00       5.00
25%    3.337000e+03       394.00      48.00      38.00
50%    2.945950e+04      3223.00     393.00     314.50
75%    2.622770e+05     30864.50    3716.50    3011.00
max    1.012000e+08  11300000.00  255800.00  908100.00

Muestra del catalogo multi-plataforma (TikTok e Instagram Reels):
     plataforma          id_referencia                  titulo             autor  tempo_bpm  popularidad
         TikTok 6kVuF2PYLuvl9T85XjNbaO Lay It Down Gmix - Main             Lloyd    155.932         28.0
         TikTok 1RGIjMFMgJxkZHMDXVYzOJ  Bartender (feat. Akon)  

## 2. Conclusión Ejecutiva (Fase 2)
1. **Tolerancia a fallos comprobada:** El cliente `src/ingesta/cliente_ytdlp.py` y el validador `src/ingesta/validador_medios.py` interceptan errores de red y descartan archivos corruptos o inferiores a 2 segundos sin interrumpir la ejecución por lotes.
2. **Integridad contractual:** El archivo `datos/procesados/matriz_acustica_cruda.parquet` consolida 650 registros únicos con 33 columnas (`id_publicacion`, 4 variables sociales y 28 variables acústicas en `float64`), con cero valores perdidos, cumpliendo el contrato de entrada hacia la Fase 4.
